In [ ]:
### DataSet - https://www.kaggle.com/code/jillanisofttech/twitter-us-airline-sentiment-analysis-96-acc/input

In [5]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense,Input
from sklearn.model_selection import train_test_split

In [6]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("punkt_tab")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [7]:
df=pd.read_csv("Tweets.csv")

In [8]:
df=df[["text","airline_sentiment"]]

In [9]:
df

,text,airline_sentiment
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative
...,...,...
14635,@AmericanAir thank you we got on a different f...,positive
14636,@AmericanAir leaving over 20 minutes Late Flig...,negative
14637,@AmericanAir Please bring American Airlines to...,neutral
14638,"@AmericanAir you have my money, you change my ...",negative


In [10]:
import re
def clean(text):
  text=text.lower()
  text=re.sub(r"http\S+|www\S+|https\S+","",text,flags=re.MULTILINE)
  text=re.sub(r"\@\w+|\#","",text)
  words=text.split()

  words=[word for word in words if word not in stopwords.words("english")]
  lemmatizer=WordNetLemmatizer()
  words=[lemmatizer.lemmatize(word) for word in words]
  return " ".join(words)



In [11]:
df["processed_text"]=df['text'].apply(clean)

In [12]:
df

,text,airline_sentiment,processed_text
0,@VirginAmerica What @dhepburn said.,neutral,said.
1,@VirginAmerica plus you've added commercials t...,positive,plus added commercial experience... tacky.
2,@VirginAmerica I didn't today... Must mean I n...,neutral,today... must mean need take another trip!
3,@VirginAmerica it's really aggressive to blast...,negative,"really aggressive blast obnoxious ""entertainme..."
4,@VirginAmerica and it's a really big bad thing...,negative,really big bad thing
...,...,...,...
14635,@AmericanAir thank you we got on a different f...,positive,thank got different flight chicago.
14636,@AmericanAir leaving over 20 minutes Late Flig...,negative,leaving 20 minute late flight. warning communi...
14637,@AmericanAir Please bring American Airlines to...,neutral,please bring american airline blackberry10
14638,"@AmericanAir you have my money, you change my ...",negative,"money, change flight, answer phones! suggestio..."


In [13]:
tokenize=Tokenizer(num_words=5000)

In [14]:
tokenize.fit_on_texts(df["processed_text"])

In [15]:
len(tokenize.word_counts)

13357

In [16]:
padded=pad_sequences(tokenize.texts_to_sequences(df["processed_text"]),maxlen=50,padding="post",truncating="post")


In [17]:
df['airline_sentiment_reviwed']=df['airline_sentiment'].map({"negative":0,"neutral":1,"positive":2})

In [18]:
df

,text,airline_sentiment,processed_text,airline_sentiment_reviwed
0,@VirginAmerica What @dhepburn said.,neutral,said.,1
1,@VirginAmerica plus you've added commercials t...,positive,plus added commercial experience... tacky.,2
2,@VirginAmerica I didn't today... Must mean I n...,neutral,today... must mean need take another trip!,1
3,@VirginAmerica it's really aggressive to blast...,negative,"really aggressive blast obnoxious ""entertainme...",0
4,@VirginAmerica and it's a really big bad thing...,negative,really big bad thing,0
...,...,...,...,...
14635,@AmericanAir thank you we got on a different f...,positive,thank got different flight chicago.,2
14636,@AmericanAir leaving over 20 minutes Late Flig...,negative,leaving 20 minute late flight. warning communi...,0
14637,@AmericanAir Please bring American Airlines to...,neutral,please bring american airline blackberry10,1
14638,"@AmericanAir you have my money, you change my ...",negative,"money, change flight, answer phones! suggestio...",0


In [19]:
model=Sequential()

model.add(Input(shape=(50,)))
model.add(Embedding(5000,128,input_shape=50))
model.add(LSTM(128,return_sequences=True))
model.add(LSTM(64))
model.add(Dense(3,activation="softmax"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [20]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 50, 128)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 50, 128)        │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 821,187 (3.13 MB)

 Trainable params: 821,187 (3.13 MB)

 Non-trainable params: 0 (0.00 B)

In [21]:
model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [22]:
labels=df["airline_sentiment_reviwed"]

In [23]:
history=model.fit(padded,labels,epochs=100,validation_split=0.2)

Epoch 1/100
366/366 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.6048 - loss: 0.9457 - val_accuracy: 0.7128 - val_loss: 0.8174
Epoch 2/100
366/366 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.6052 - loss: 0.9405 - val_accuracy: 0.7128 - val_loss: 0.8098
Epoch 3/100
366/366 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.6054 - loss: 0.9425 - val_accuracy: 0.7128 - val_loss: 0.8309
Epoch 4/100
366/366 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.6054 - loss: 0.9420 - val_accuracy: 0.7128 - val_loss: 0.8153
Epoch 5/100
366/366 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.6054 - loss: 0.9421 - val_accuracy: 0.7128 - val_loss: 0.8325
Epoch 6/100
366/366 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.6054 - loss: 0.9420 - val_accuracy: 0.7128 - val_loss: 0.8144
Epoch 7/100
366/366 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.6054 - loss: 0.9419 - val_accuracy: 0.7128 - val_loss: 0.8153
Epoch 8/100
366/366 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.6054 - loss: 0.9417 - va

In [24]:
text=df["text"][3]

In [26]:
text=clean(text)

In [27]:
text=tokenize.texts_to_sequences([text])

In [28]:
text=pad_sequences(text,maxlen=50,padding="post",truncating="post")

In [30]:
y_pred=model.predict(text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step


In [31]:
np.argmax(y_pred,axis=1)

array([0])